# What drives the forecast

We check why the exposure map follows distance from the crater more than the plume wedge, using the saved models
and the outputs of notebooks 01 to 03. Nothing here retrains a model or changes an earlier result. Outputs:

1. `results/tables/shap_feature_groups_2026.csv`: mean absolute SHAP per feature group, 2026
2. `results/tables/wedge_effect_2026.csv`: p_exposed inside and outside the 45-degree plume wedge, by distance
3. `results/tables/report_drift_alignment.csv`: reported barangays near the observed and modeled drift
4. `results/tables/plume_height_report_days.csv`: PHIVOLCS plume heights on report days and other days
5. `results/tables/upper_wind_check.csv`: ashfall reports against the wind at 800-700, 600 and 500 hPa
6. `results/forecast_drivers.json`: the numbers shown on the dashboard's Methods & results page

## Setup

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd() if (Path.cwd() / "requirements.txt").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src import forecast, geo

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORT_LOG = ROOT / "data" / "report_log" / "report_log.csv"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"

WEDGE_HALF_DEG = 22.5  # the dashboard's plume wedge is 45 degrees wide
NEAR_DEG = 45
UPWIND_DEG = 90
DIST_BANDS_KM = [0, 10, 20, 30]
MODEL_FILE_SUFFIX = {"MLP": "mlp", "LightGBM": "lgbm"}
FEATURE_GROUPS = {
    "fixed geography": ["dist_km", "bearing_sin", "bearing_cos", "elev_m"],
    "wind and plume": ["angle_diff_deg", "log_plume_index", "wind_speed_plume"],
    "earthquakes": forecast.EQ_EXPOSURE_COLUMNS,
    "SO2 and plume height": ["log_so2_input", "plume_height_m"],
}
HISTORICAL_URL = "https://historical-forecast-api.open-meteo.com/v1/forecast"
UPPER_LEVELS = ["600hPa", "500hPa"]
UPPER_WIND_FILE = RAW / "wind_kanlaon_upper_report_days.csv"

pd.set_option("display.width", 160)


def show(df, name):
    """Print row count and missing values per column."""
    print(f"{name}: {len(df):,} rows x {df.shape[1]} columns")
    na = df.isna().sum()
    na = na[na > 0]
    print("  missing:", "none" if na.empty else na.to_dict())

## Load data

In [2]:
metrics = json.loads((RESULTS / "metrics.json").read_text())
volcano = pd.read_csv(PROCESSED / "volcano_daily.csv", parse_dates=["date"]).set_index("date")
print(f"volcano_daily: {len(volcano):,} rows x {volcano.shape[1]} columns, "
      f"{volcano.isna().any().sum()} columns with missing values")

PANEL_COLUMNS = ["adm4_psgc", "date", "split", "report_day", "angle_diff_deg", "dist_km", "bearing_deg", "elev_m",
                 "wind_speed_plume", *forecast.EQ_EXPOSURE_COLUMNS]
panel = pd.read_csv(PROCESSED / "barangay_daily_panel.csv", usecols=PANEL_COLUMNS, parse_dates=["date"])
show(panel, "barangay_daily_panel")
scores = (pd.read_csv(PROCESSED / "exposure_scores.csv", usecols=["psgc_code", "date", "plume_index", "p_exposed"],
                      parse_dates=["date"])
          .rename(columns={"psgc_code": "adm4_psgc"}))
show(scores, "exposure_scores")
panel = panel.merge(scores, on=["adm4_psgc", "date"], how="left", validate="1:1")
show(panel, "panel + exposure scores")

barangays = pd.read_csv(PROCESSED / "barangays_study_area.csv", usecols=["adm4_psgc", "bearing_deg", "dist_km"])
show(barangays, "barangays_study_area")
report_log = pd.read_csv(REPORT_LOG, usecols=["event_date", "psgc_code", "hazard"], parse_dates=["event_date"])
report_log["psgc_code"] = report_log["psgc_code"].astype("Int64")
show(report_log, "report_log")
report_dates = pd.DatetimeIndex(sorted(report_log["event_date"].unique()))
print("report days:", len(report_dates))

volcano_daily: 1,025 rows x 81 columns, 69 columns with missing values


barangay_daily_panel: 221,400 rows x 13 columns
  missing: {'angle_diff_deg': 216, 'wind_speed_plume': 216, 'eq_total_lag1': 1944, 'eq_total_lag2': 2160, 'eq_total_lag3': 2376}
exposure_scores: 221,400 rows x 4 columns
  missing: {'plume_index': 216}
panel + exposure scores: 221,400 rows x 15 columns
  missing: {'angle_diff_deg': 216, 'wind_speed_plume': 216, 'eq_total_lag1': 1944, 'eq_total_lag2': 2160, 'eq_total_lag3': 2376, 'plume_index': 216}
barangays_study_area: 216 rows x 3 columns
  missing: none
report_log: 243 rows x 3 columns
  missing: {'psgc_code': 1}
report days: 14


## SHAP by feature group, 2026

The 2026 inputs are rebuilt as in notebook 02 (SO2 forecast from the final SO2 model, same-day plume height), and the
final exposure model must reproduce the saved p_exposed. SHAP values come from LightGBM's built-in TreeSHAP.

In [3]:
so2_model = joblib.load(MODELS / f"so2_next_day_{MODEL_FILE_SUFFIX[metrics['so2_next_day']['forecast_model']]}.joblib")
exposure_model = joblib.load(MODELS / f"exposure_{MODEL_FILE_SUFFIX[metrics['exposure']['scoring_model']]}.joblib")
so2_forecast = pd.Series(forecast.forecast_so2_next_day(so2_model, volcano, volcano.index),
                         index=volcano.index + pd.Timedelta(days=1))
plume_height = volcano["plume_height_median"].fillna(metrics["plume_index"]["train_median_plume_height_m"])

test = panel[panel["split"] == "test"].copy()
test["so2_input_td"] = test["date"].map(so2_forecast)
test["plume_height_m"] = test["date"].map(plume_height)
bearing_rad = np.radians(test["bearing_deg"])
test["bearing_sin"] = np.sin(bearing_rad)
test["bearing_cos"] = np.cos(bearing_rad)
test["log_plume_index"] = np.log1p(test["plume_index"])
test["log_so2_input"] = np.log1p(test["so2_input_td"])
show(test[forecast.EXPOSURE_FEATURES], "2026 exposure inputs")
gap = np.abs(exposure_model.predict_proba(test[forecast.EXPOSURE_FEATURES])[:, 1] - test["p_exposed"]).max()
print(f"largest difference from the saved p_exposed: {gap:.1e}")
assert gap < 1e-9

contributions = pd.DataFrame(exposure_model.predict(test[forecast.EXPOSURE_FEATURES], pred_contrib=True)[:, :-1],
                             columns=forecast.EXPOSURE_FEATURES)
mean_abs = contributions.abs().mean()
shap_groups = pd.DataFrame({"features": [", ".join(f) for f in FEATURE_GROUPS.values()],
                            "mean_abs_shap": [mean_abs[f].sum() for f in FEATURE_GROUPS.values()]},
                           index=pd.Index(list(FEATURE_GROUPS), name="group"))
shap_groups["share"] = shap_groups["mean_abs_shap"] / mean_abs.sum()
print("Mean absolute SHAP per feature, 2026 (log-odds)")
print(mean_abs.sort_values(ascending=False).round(3).to_string())
shap_groups.round(3)

2026 exposure inputs: 63,504 rows x 13 columns
  missing: {'angle_diff_deg': 216, 'log_plume_index': 216, 'wind_speed_plume': 216, 'eq_total_lag1': 216, 'eq_total_lag2': 216, 'eq_total_lag3': 216}
largest difference from the saved p_exposed: 1.1e-16


Mean absolute SHAP per feature, 2026 (log-odds)
dist_km             1.218
bearing_cos         0.886
eq_total_lag1       0.596
angle_diff_deg      0.461
bearing_sin         0.439
elev_m              0.413
log_plume_index     0.412
wind_speed_plume    0.253
eq_total_lag3       0.230
plume_height_m      0.228
log_so2_input       0.129
eq_total_roll7      0.060
eq_total_lag2       0.040


,features,mean_abs_shap,share
group,,,
fixed geography,"dist_km, bearing_sin, bearing_cos, elev_m",2.957,0.551
wind and plume,"angle_diff_deg, log_plume_index, wind_speed_plume",1.126,0.210
earthquakes,"eq_total_lag1, eq_total_lag2, eq_total_lag3, e...",0.926,0.173
SO2 and plume height,"log_so2_input, plume_height_m",0.356,0.066


## Plume wedge effect, 2026

A barangay is inside the wedge when its direction from the crater is within 22.5 degrees of that day's modeled
plume-level wind drift, the same angle the exposure model reads.

In [4]:
with_wind = panel[(panel["split"] == "test") & panel["angle_diff_deg"].notna()].copy()
with_wind["inside_wedge"] = with_wind["angle_diff_deg"] <= WEDGE_HALF_DEG
with_wind["dist_band"] = pd.cut(with_wind["dist_km"], DIST_BANDS_KM, right=False,
                                labels=[f"{a}-{b} km" for a, b in zip(DIST_BANDS_KM[:-1], DIST_BANDS_KM[1:])])
show(with_wind[["p_exposed", "inside_wedge", "dist_band"]], "2026 barangay-days with wind")


def wedge_row(rows):
    """Mean p_exposed inside and outside the wedge, with counts and the ratio."""
    inside, outside = rows.loc[rows["inside_wedge"], "p_exposed"], rows.loc[~rows["inside_wedge"], "p_exposed"]
    return pd.Series({"n_inside": len(inside), "mean_p_inside": inside.mean(), "n_outside": len(outside),
                      "mean_p_outside": outside.mean(), "ratio": inside.mean() / outside.mean()})


wedge_effect = pd.concat([
    with_wind.groupby("dist_band", observed=True).apply(wedge_row),
    wedge_row(with_wind[with_wind["dist_km"] >= 10]).rename("10-30 km").to_frame().T,
    wedge_row(with_wind).rename("all").to_frame().T,
]).rename_axis("dist_band")
print(f"2026 days with wind: {with_wind['date'].nunique()}")
wedge_effect.round(3)

2026 barangay-days with wind: 63,288 rows x 3 columns
  missing: none
2026 days with wind: 293


,n_inside,mean_p_inside,n_outside,mean_p_outside,ratio
dist_band,,,,,
0-10 km,657.0,0.903,3738.0,0.506,1.785
10-20 km,2565.0,0.544,16187.0,0.156,3.489
20-30 km,6137.0,0.106,34004.0,0.047,2.228
10-30 km,8702.0,0.235,50191.0,0.082,2.850
all,9359.0,0.282,53929.0,0.112,2.521


## Reported barangays and the plume direction

Each reported study barangay-day is compared with the PHIVOLCS drift observed that day (daily mode, 8 sectors) and
the modeled plume-level wind drift from notebook 01.

In [5]:
reports = report_log[report_log["psgc_code"].isin(barangays["adm4_psgc"])]
reports = (reports.assign(ashfall=reports["hazard"].str.contains("ashfall"),
                          sulfur_odor=reports["hazard"].str.contains("sulfur odor"))
           .groupby(["psgc_code", "event_date"], as_index=False)[["ashfall", "sulfur_odor"]].max()
           .astype({"psgc_code": "int64"})
           .merge(barangays.rename(columns={"adm4_psgc": "psgc_code"}), on="psgc_code", how="left", validate="m:1"))
reports["observed_drift_deg"] = reports["event_date"].map(volcano["drift_obs_sector"]).map(geo.SECTOR_DEG)
reports["modeled_drift_deg"] = reports["event_date"].map(volcano["wind_drift_deg"])
show(reports, "reported study barangay-days")


def alignment_row(angles):
    """Shares of reported barangays inside the wedge, within 45 degrees and upwind of a drift direction."""
    angles = angles.dropna()
    return {"barangay_days": len(angles), "share_inside_wedge": (angles <= WEDGE_HALF_DEG).mean(),
            "share_within_45deg": (angles <= NEAR_DEG).mean(), "share_upwind": (angles >= UPWIND_DEG).mean()}


rows = []
for reference, column in [("observed PHIVOLCS drift", "observed_drift_deg"),
                          ("modeled wind drift (800-700 hPa)", "modeled_drift_deg")]:
    angles = pd.Series(geo.angle_diff_deg(reports["bearing_deg"], reports[column]), index=reports.index)
    for hazard, mask in [("all reports", reports.index == reports.index), ("ashfall", reports["ashfall"]),
                         ("sulfur odor", reports["sulfur_odor"])]:
        rows.append({"reference": reference, "hazard": hazard, **alignment_row(angles[mask])})
report_alignment = pd.DataFrame(rows)
report_alignment.round(3)

reported study barangay-days: 166 rows x 8 columns
  missing: {'observed_drift_deg': 24}


,reference,hazard,barangay_days,share_inside_wedge,share_within_45deg,share_upwind
0,observed PHIVOLCS drift,all reports,142,0.345,0.563,0.169
1,observed PHIVOLCS drift,ashfall,95,0.411,0.653,0.084
2,observed PHIVOLCS drift,sulfur odor,80,0.288,0.438,0.300
3,modeled wind drift (800-700 hPa),all reports,166,0.470,0.669,0.120
4,modeled wind drift (800-700 hPa),ashfall,111,0.613,0.793,0.045
5,modeled wind drift (800-700 hPa),sulfur odor,94,0.287,0.489,0.202


## Plume heights on report days

Heights are above the crater (2,435 m). The daily median sets the plume altitude for the wind; the daily maximum
shows how high the column reached.

In [6]:
volcano["report_day"] = volcano.index.isin(report_dates)
plume_heights = (volcano.groupby("report_day")
                 .agg(days_with_height=("plume_height_median", "count"),
                      median_of_daily_median_m=("plume_height_median", "median"),
                      median_of_daily_max_m=("plume_height_max", "median"),
                      highest_daily_max_m=("plume_height_max", "max"))
                 .rename(index={True: "report days", False: "other days"}))
plume_heights.index.name = "day_type"
print(volcano.loc[volcano["report_day"], ["plume_height_median", "plume_height_max", "drift_obs_sector"]].to_string())
plume_heights

            plume_height_median  plume_height_max drift_obs_sector
date                                                              
2024-06-03               3000.0            5000.0                S
2024-09-11                  NaN               NaN              NaN
2024-09-24                100.0             100.0               SW
2024-10-28                  NaN               NaN              NaN
2024-11-15                350.0             750.0                W
2024-12-09               2050.0            4000.0               SW
2025-02-15               1100.0            1500.0               NW
2025-04-08                200.0            4000.0               SW
2025-10-24                 75.0              75.0               NW
2026-02-19                 37.5            2000.0               SW
2026-02-26                500.0            2500.0               SW
2026-03-15                400.0            5000.0               SE
2026-07-09                  NaN               NaN             

,days_with_height,median_of_daily_median_m,median_of_daily_max_m,highest_daily_max_m
day_type,,,,
other days,872,200.0,300.0,2000.0
report days,11,400.0,2500.0,6000.0


## Upper-level wind on report days

Ash columns rise far above the 700 hPa level that bounds our plume wind. We download hourly wind at 600 and
500 hPa for the 14 report days from the same Open-Meteo historical forecast API as `wind_kanlaon_hourly.csv`
(saved once to `data/raw/wind_kanlaon_upper_report_days.csv`) and average u and v from 06:00 to 18:00.

In [7]:
LEVELS = ["800hPa", "700hPa", *UPPER_LEVELS]
HOURLY = [f"{variable}_{level}" for level in LEVELS
          for variable in ("wind_speed", "wind_direction", "geopotential_height")]
if not UPPER_WIND_FILE.exists():
    parts = []
    for day in report_dates:
        response = requests.get(HISTORICAL_URL, timeout=60, params={
            "latitude": geo.CRATER_LAT, "longitude": geo.CRATER_LON, "start_date": f"{day:%Y-%m-%d}",
            "end_date": f"{day:%Y-%m-%d}", "hourly": ",".join(HOURLY), "wind_speed_unit": "ms",
            "timezone": "Asia/Manila"})
        response.raise_for_status()
        parts.append(pd.DataFrame(response.json()["hourly"]))
    pd.concat(parts, ignore_index=True).to_csv(UPPER_WIND_FILE, index=False)
upper = pd.read_csv(UPPER_WIND_FILE, parse_dates=["time"])
show(upper, "wind_kanlaon_upper_report_days")

# the 800 and 700 hPa columns must match the wind file used by notebooks 01 and 02
hourly_wind = pd.read_csv(RAW / "wind_kanlaon_hourly.csv", parse_dates=["time"])
same = upper.merge(hourly_wind, on="time", suffixes=("", "_file"))
print(f"hours matched with wind_kanlaon_hourly.csv: {len(same)} of {len(upper)}")
for column in ["wind_speed_800hPa", "wind_direction_800hPa", "wind_speed_700hPa", "wind_direction_700hPa"]:
    print(f"  {column}: max difference {(same[column] - same[f'{column}_file']).abs().max()}")

wind_kanlaon_upper_report_days: 336 rows x 13 columns
  missing: none
hours matched with wind_kanlaon_hourly.csv: 336 of 336
  wind_speed_800hPa: max difference 0.0
  wind_direction_800hPa: max difference 0
  wind_speed_700hPa: max difference 0.0
  wind_direction_700hPa: max difference 0


In [8]:
daytime = upper[upper["time"].dt.hour.between(*forecast.DAYTIME_HOURS)].assign(date=lambda d: d["time"].dt.normalize())
level_drift = pd.DataFrame(index=report_dates)
for level in UPPER_LEVELS:
    u, v = geo.wind_to_uv(daytime[f"wind_speed_{level}"], daytime[f"wind_direction_{level}"])
    g = pd.DataFrame({"date": daytime["date"], "u": u, "v": v}).groupby("date")
    level_drift[level] = (geo.uv_to_wind(g["u"].mean(), g["v"].mean())[1] + 180) % 360
level_drift["800-700hPa"] = volcano.loc[report_dates, "wind_drift_deg"]
altitude_m = {"800-700hPa": None, **{level: daytime[f"geopotential_height_{level}"].mean() for level in UPPER_LEVELS}}

ash = reports[reports["ashfall"]]
per_day = ash.groupby("event_date").size().rename("ashfall_barangays").to_frame()
rows = []
for level in ["800-700hPa", *UPPER_LEVELS]:
    angles = geo.angle_diff_deg(ash["bearing_deg"], ash["event_date"].map(level_drift[level]))
    near = pd.Series(angles <= NEAR_DEG, index=ash.index)
    per_day[f"drift_{level}"] = geo.deg_to_sector(level_drift.loc[per_day.index, level])
    per_day[f"within45_{level}"] = near.groupby(ash["event_date"]).sum()
    rows.append({"wind_level": level, "altitude_m": altitude_m[level],
                 "report_days": ash["event_date"].nunique(), "ashfall_barangay_days": len(ash),
                 "within_45deg": int(near.sum()), "share_within_45deg": near.mean()})
upper_wind_check = pd.DataFrame(rows)
print(per_day.to_string())
upper_wind_check.round(3)

            ashfall_barangays drift_800-700hPa  within45_800-700hPa drift_600hPa  within45_600hPa drift_500hPa  within45_500hPa
event_date                                                                                                                     
2024-11-15                  5                W                    3           SW                3           SW                3
2024-12-09                  2               NW                    2           NW                2           NW                2
2025-02-15                  9               NW                    9           NW                9           NW                9
2025-04-08                  4                W                    4            W                4            W                4
2025-10-24                 13               NW                    9           NW                8            N                8
2026-02-19                 27                W                   24            W               22       

,wind_level,altitude_m,report_days,ashfall_barangay_days,within_45deg,share_within_45deg
0,800-700hPa,NaN,10,111,88,0.793
1,600hPa,4414.209,10,111,78,0.703
2,500hPa,5873.841,10,111,62,0.559


In [9]:
current = upper_wind_check.set_index("wind_level")["share_within_45deg"]
best_upper = current[UPPER_LEVELS].idxmax()
upper_matches = bool(current[best_upper] >= current["800-700hPa"])
print(f"800-700 hPa {current['800-700hPa']:.1%}, 600 hPa {current['600hPa']:.1%}, 500 hPa {current['500hPa']:.1%}")
print(f"best upper level: {best_upper}; matches ashfall reports at least as well as our wind: {upper_matches}")

800-700 hPa 79.3%, 600 hPa 70.3%, 500 hPa 55.9%
best upper level: 600hPa; matches ashfall reports at least as well as our wind: False


## Save

In [10]:
shap_groups.to_csv(TABLES / "shap_feature_groups_2026.csv", float_format="%.4f")
wedge_effect.to_csv(TABLES / "wedge_effect_2026.csv", float_format="%.4f")
report_alignment.to_csv(TABLES / "report_drift_alignment.csv", index=False, float_format="%.4f")
plume_heights.to_csv(TABLES / "plume_height_report_days.csv")
upper_wind_check.to_csv(TABLES / "upper_wind_check.csv", index=False, float_format="%.4f")

modeled_all = report_alignment.set_index(["reference", "hazard"]).loc[("modeled wind drift (800-700 hPa)",
                                                                      "all reports")]
drivers = {
    "shap_group_share_2026": shap_groups["share"].round(4).to_dict(),
    "wedge_effect_2026": wedge_effect[["mean_p_inside", "mean_p_outside", "ratio"]].round(4).to_dict(orient="index"),
    "reports_modeled_wedge": {"barangay_days": int(modeled_all["barangay_days"]),
                              "share_inside_wedge": round(float(modeled_all["share_inside_wedge"]), 4),
                              "share_within_45deg": round(float(modeled_all["share_within_45deg"]), 4)},
    "plume_height_m": plume_heights.astype(float).to_dict(orient="index"),
    "upper_wind": {"share_within_45deg": current.round(4).to_dict(),
                   "altitude_m": {k: round(v) for k, v in altitude_m.items() if v is not None},
                   "ashfall_barangay_days": int(len(ash)), "report_days": int(ash["event_date"].nunique()),
                   "best_upper_level": best_upper, "upper_matches_at_least_as_well": upper_matches},
}
(RESULTS / "forecast_drivers.json").write_text(json.dumps(drivers, indent=2))
print("saved 5 tables and forecast_drivers.json")

saved 5 tables and forecast_drivers.json
